# Improved Neural Network to Diagnose Diabetes

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy import stats
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager

df = pd.read_csv("../data/diabetes_dataset.csv")

In [2]:
df_clean = df.drop(columns=[
    'diabetes_stage', 'diabetes_risk_score'])

In [3]:
# split features (X) and target (y)
X = df_clean.drop('diagnosed_diabetes', axis=1)
X = pd.get_dummies(X, drop_first=True)
y = df_clean['diagnosed_diabetes']

# test cases
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

# create train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# standardize numerical features using StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Shape of X: (100000, 40)
Shape of y: (100000,)


In [4]:
# convert the numpy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# create DataLoaders for batch processing
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [5]:
class DiabetesNN(nn.Module):
    def __init__(self, input_dim):
        super(DiabetesNN, self).__init__()
        # first layer is input to hidden
        self.layer1 = nn.Linear(input_dim, 256)
        # second is hidden to hidden
        self.layer2 = nn.Linear(256, 128)
        # third is hidden to hidden
        self.layer3 = nn.Linear(128, 64)
        # fourth is hidden to output
        self.layer4 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(0.2)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.layer2(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.layer3(x)
        x = self.relu(x)

        x = self.layer4(x)
        x = self.sigmoid(x)
        return x

In [6]:
# initialize model
input_dim = X.shape[1]
model_sgd = DiabetesNN(X.shape[1])

# define optimizer and loss
criterion = nn.BCELoss()

optimizer_sgd = optim.SGD(model_sgd.parameters(), lr=0.01, momentum=0.9)

num_epochs = 50

print("Training for Non-DP SGD")
for epoch in range(num_epochs):
    model_sgd.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer_sgd.zero_grad()
        outputs = model_sgd(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_sgd.step()
        
        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

# Print out the accuracy:
model_sgd.eval()
with torch.no_grad():
    outputs = model_sgd(X_test_tensor)
    predicted = (outputs > 0.5).float()
    accuracy = (predicted == y_test_tensor).sum() / y_test_tensor.size(0)
    print(f"\nStandard SGD Test Accuracy: {accuracy.item() * 100:.2f}%")

Training for Non-DP SGD
Epoch 10, Loss: 0.2431
Epoch 20, Loss: 0.2301
Epoch 30, Loss: 0.2227
Epoch 40, Loss: 0.2183
Epoch 50, Loss: 0.2140

Standard SGD Test Accuracy: 91.25%


In [7]:
def train_dp_model(noise_multiplier, max_grad_norm, epochs, delta):
    # initialize the neural network and criterion
    model_dp = DiabetesNN(X.shape[1])
    optimizer = optim.SGD(model_dp.parameters(), lr=0.05, momentum=0.9)
    criterion = nn.BCELoss()

    # recreate loader and initialize privacy engine
    train_loader_dp = DataLoader(train_dataset, batch_size=64, shuffle=True)
    privacy_engine = PrivacyEngine()

    # make the model private
    # wrap the model, optimizer, and loader to handle the gradient clipping and noise addition
    # everything else after this is the same
    model, optimizer, train_loader_dp = privacy_engine.make_private(
        module = model_dp,
        optimizer = optimizer,
        data_loader = train_loader_dp,
        noise_multiplier = noise_multiplier,
        max_grad_norm = max_grad_norm,
    )

    # starting the training
    # printing out the noise multiplier at the top
    print(f"\nTraining with noise {noise_multiplier}\n")
    # same as before
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader_dp:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

    # display epsilon and accuracy
    epsilon = privacy_engine.get_epsilon(delta=delta)
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        predicted = (outputs > 0.5).float()
        accuracy = (predicted == y_test_tensor).float().sum() / y_test_tensor.size(0)
        
    return epsilon, accuracy.item() * 100

In [8]:
noise_params = [0.45, 0.5, 0.7, 0.9, 1.0, 1.5] 
max_grad_norm = 1.0
results = []

print("Starting DP Training")

for nm in noise_params:
    # run epochs
    eps, acc = train_dp_model(noise_multiplier=nm, max_grad_norm=max_grad_norm, epochs=20, delta=1e-5)
    results.append((nm, eps, acc))

# display in a table format
print(f"{'Noise Mult':<15} {'Epsilon':<15} {'Accuracy (%)':<15}")
for nm, eps, acc in results:
    print(f"{nm:<15} {eps:<15.2f} {acc:<15.2f}")

Starting DP Training

Training with noise 0.45



/Users/shamusmurphy/.pyenv/versions/3.12.0/lib/python3.12/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/var/folders/lp/x6g4gh1x47sb4m1nrn0p2cwr0000gn/T/ipykernel_12024/2858078453.py:34: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()



Training with noise 0.5


Training with noise 0.7


Training with noise 0.9


Training with noise 1.0


Training with noise 1.5

Noise Mult      Epsilon         Accuracy (%)   
0.45            9.30            85.97          
0.5             5.91            81.43          
0.7             1.45            73.47          
0.9             0.75            60.87          
1.0             0.62            59.35          
1.5             0.33            57.21          
